# Job Listing Prioritization for Career Changers: Profiles (career switches)

PredictLeads × TripleTen externship · Jacob Colombo

## Purpose

Finds **real career changers** (anyone switching from their current background to a different role or field) in a public resume dataset: people who moved from another field into a **computer and mathematical** role, or from one tech role to another. It feeds two things:
- **The retraining compass, past side** (`DIFFERENTIATORS.md`): prior career → the training people took → their first tech job (4a.2), and tech job before → after a move within tech, sideways or up (4a.3). EDA_SDA_2 adds each role's 2026 openings, pay, closing speed and expected growth.
- **The role model** (Option B, `docs/fit_analysis.md`): each career changer gives a **pre-switch profile** (jobs and education before the switch) and a **real label** (the role they moved into).

**How a career changer is found.** Each resume is read as one timeline: jobs and education in date order, each entry labelled tech/math, other field, unclear, or skip (internships, high school). The **original background** is everything before the first tech or math entry. **Nobody who reached a computer and mathematical job is thrown away:** each person gets labels (origin, retrained, first tech job level, every tech role reached), and `status` names the main one. **Career changers are a view,** in three pools: **retrained after working in another field** and **retrained as a new grad from another field** (another field, then a tech or math program, then the tech job), and **changed roles within tech** (a move into a different tech role bucket after 12+ months in the first). Any job level counts, including manager. The ML model can learn from everyone, with origin as a feature; results are reported on career changers. The rules live in `src/career_switch.py`.

**Data:** Kaggle `suriyaganesh/resume-dataset-structured` (MIT license, version 2 of 2024-11-14): about 54,000 resume records, converted by OCR and an LLM, so expect extraction errors. Stored in git-ignored `data/external/resume_structured/`.

**Data rules:** names and contact details (`people.csv`), employers and locations are never loaded. Aggregates only, except the cells tagged `data-rows` (the 4a.2d and 5b previews, the 4a.2e title list and the 4c review sample): clear them before committing. Never paste rows into AI tools.

**Leakage rule:** the pre-switch profile uses only jobs and education that started before the switch. Undated programs are placed by degree level (flagged in `study_undated`). Skills aren't dated, so they stay out of the pre-switch profile.

### Clarifying questions
- How many people switched into a computer and mathematical role, by retraining or by moving within tech?
- From each prior career, which first tech jobs did people reach, and through which training? (The compass for changing careers.)
- Within tech, which moves do people make, and how many are a step up rather than sideways? (The compass for moving across or up.)
- How often do the rules mistake a title change, a student job or a technical background for a career change?

### Roadmap
1. Setup · 2. Load, then remove duplicate resumes (2b) · 3. Dates and title flags
4. Find the switches; viable candidates (4a.1); retrained: prior career, training and first tech job (4a.2); changed roles within tech, sideways or up (4a.3); the review round (4b, 4c)
5. Freeze and save: the modeling table · 6. Conclusion

## 1. Setup

In [ ]:
# 1a Libraries, shared helpers and the career-switch rules
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "requirements.txt").exists())
sys.path.insert(0, str(ROOT))  # so `src` can be imported from notebooks/
from scipy import stats
from sklearn.linear_model import LogisticRegression

from src.eda_helpers import BLUE, DATA_DIR, INK_2, ORANGE, bar_h, finish, heatmap, show, use_chart_style
from src.career_switch import (FIELD_ORDER, ROLE_ORDER, RULES, background_area, find_switches, path_table, person_timeline,
                               prepare_education, tag_jobs)
use_chart_style()

RESUME_DIR = DATA_DIR / "external" / "resume_structured"
PROFILES_DIR = DATA_DIR / "profiles"  # saved tables for the ML notebook
PROFILES_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load

Only the columns the rules need. If the dataset's headers differ from its documentation, this stops and lists what's missing.

In [ ]:
# 2a Find the files and load only the columns we need
import re
import zipfile

def is_named(path, name):
    """Match a file name, ignoring case and a leading number prefix (the zip has 04_experience.csv)."""
    return re.sub(r"^\d+_", "", Path(path).name.lower()) == name.lower()

def unzip_if_needed():
    """If the CSVs aren't unzipped yet, extract the Kaggle zip found anywhere under data/ (it must contain experience.csv)."""
    if any(is_named(p, "experience.csv") for p in RESUME_DIR.rglob("*")):
        return
    zips = sorted(DATA_DIR.rglob("*.zip"))
    for zp in zips:
        with zipfile.ZipFile(zp) as z:
            names = z.namelist()
            if not any(is_named(n, "experience.csv") for n in names):
                continue
            if any(Path(n).is_absolute() or ".." in Path(n).parts for n in names):
                raise ValueError(f"{zp.name} has unsafe paths; not extracting")
            RESUME_DIR.mkdir(parents=True, exist_ok=True)
            z.extractall(RESUME_DIR)
        print(f"unzipped {zp.relative_to(DATA_DIR.parent)} into {RESUME_DIR.relative_to(DATA_DIR.parent)} ({len(names)} files); you can delete the zip")
        return
    # Nothing matched: report file names only (never contents), so the problem can be fixed
    print("No zip under data/ contains experience.csv. Zips found:", [str(z.relative_to(DATA_DIR.parent)) for z in zips] or "none")
    for zp in zips:
        try:
            with zipfile.ZipFile(zp) as z:
                print(f"  {zp.name}: {len(z.namelist())} files; CSVs: {[Path(n).name for n in z.namelist() if n.lower().endswith('.csv')][:20]}")
        except zipfile.BadZipFile:
            print(f"  {zp.name}: not a valid zip (still downloading, or the download failed)")

def find_csv(name):
    unzip_if_needed()
    hits = sorted(p for p in RESUME_DIR.rglob("*") if is_named(p, name))
    if not hits:
        raise FileNotFoundError(f"{name} not found under {RESUME_DIR}. Download the dataset first (see docs/fit_analysis.md).")
    return hits[0]

def load(name, cols):
    path = find_csv(name)
    header = pd.read_csv(path, nrows=0).columns
    missing = [c for c in cols if c not in header]
    if missing:
        raise KeyError(f"{name}: missing columns {missing}; found {list(header)}")
    return pd.read_csv(path, usecols=cols, dtype=str, encoding_errors="replace")

experience = load("experience.csv", ["person_id", "title", "start_date", "end_date"])
education = load("education.csv", ["person_id", "program", "start_date"])

show("Loaded tables (shape, and the share of each column filled)", pd.DataFrame({
    name: {"rows": len(df), "people": df.person_id.nunique(),
           **{f"{c} filled": round(df[c].notna().mean(), 3) for c in df.columns if c != "person_id"}}
    for name, df in {"experience": experience, "education": education}.items()
}).T)

### 2b Duplicates

The dataset combines several resume collections, and the first run's counts were almost all multiples of 3, which suggests each resume appears about three times under different `person_id`s. Duplicates would show the same person twice in a review round and put copies of one person on both sides of a train/test split (leakage).

Each person gets a **fingerprint**: every job (title, start and end date as written) plus every program (name and start date), sorted. People with the same fingerprint are the same resume; the lowest `person_id` is kept. Only **exact** copies are caught: a copy with an OCR difference gets a different fingerprint.

In [ ]:
# 2b Fingerprint every resume and keep one copy of each
from src.career_switch import fingerprints

DEDUPE = True
fp = fingerprints(experience, education)
N_BEFORE_DEDUPE = len(fp)
if DEDUPE:
    order = np.argsort(pd.to_numeric(fp.index, errors="coerce"), kind="stable")
    keep = fp.iloc[order].drop_duplicates().index  # the lowest person_id of each fingerprint
    experience = experience[experience.person_id.isin(keep)]
    education = education[education.person_id.isin(keep)]

show("Duplicate resumes", pd.Series({
    "resumes in the dataset": N_BEFORE_DEDUPE,
    "duplicates removed": N_BEFORE_DEDUPE - experience.person_id.nunique(),
    "resumes left": experience.person_id.nunique(),
}).to_frame("resumes"))

**What the data shows**
- **54,933 resume records held 17,952 unique people:** 36,981 were duplicates, so each resume appeared about 3 times.

## 3. Dates and title flags

Dates become months (`MM/YYYY`; a bare year becomes January; "Present" becomes the dataset's update month, 2024-11). Titles are matched against the rules: which computer and mathematical bucket they fall in, and whether they're senior, school or internship work, or already technical. 3b shows the *patterns* of dates the parser still can't read (# for a digit, Word for letters), so the parser can be extended without looking at any dates.


In [ ]:
# 3a Parse dates and flag titles
jobs = tag_jobs(experience)
show("Computer and mathematical jobs by bucket", jobs.cm_bucket.value_counts().to_frame("jobs"))

In [ ]:
# 3b Date formats the parser can't read yet: patterns only (# = a digit, Word = letters), never the dates
def date_pattern(value):
    """'Jun 2015' -> 'Word ####', '06/15' -> '##/##'."""
    return re.sub(r"[A-Za-z]+", lambda m: "Word" if m.group(0)[0].isupper() else "word", re.sub(r"\d", "#", value.strip()))

unread = {}
for raw_col, parsed_col, label in [("start_date", "start", "start dates"), ("end_date", "end", "end dates")]:
    filled = jobs[raw_col].notna() & jobs[raw_col].astype(str).str.strip().ne("")
    unread[label] = jobs.loc[filled & jobs[parsed_col].isna(), raw_col].astype(str).map(date_pattern).value_counts()
patterns = pd.DataFrame(unread).fillna(0).astype(int)
patterns["share of all unreadable"] = (patterns.sum(axis=1) / patterns.sum().sum()).round(3)
patterns = patterns.sort_values("share of all unreadable", ascending=False).head(15).rename_axis("date pattern")
show(f"Unreadable dates by pattern (# = a digit, Word = letters): {unread['start dates'].sum():,} start dates and "
     f"{unread['end dates'].sum():,} end dates still unreadable", patterns)

## 4. Find the switches

For each person, in date order: their **first computer and mathematical job** is the switch, and the non-computer-and-mathematical jobs before it are their **background**. Everything before it is read as one timeline with their education. Their **original background** is everything before their first tech or math entry (a tech job, or a tech or math program). A person is a **candidate** if that background includes another field **and** they trained into tech or math before the switch: `candidate: retrained, new grad from another field` or `candidate: retrained, after working in another field`. Each rejection is kept with its reason, so we can review the rejected pools too.

The rules label instead of rejecting: every person who reached a computer and mathematical job stays in `switches`, with `origin`, `retrained`, `first_level` and `roles_reached` (every tech role they held, not just the first). 4a's second cell counts people by label.


In [ ]:
# 4a One row per person: candidate, or the first rule that rejected them
switches = find_switches(jobs, education)
funnel = switches.status.value_counts().rename_axis("status").to_frame("people")
funnel["share"] = (funnel.people / len(switches)).round(3)
show("People by status", funnel)

fig, ax = plt.subplots(figsize=(9, 4.2))
bar_h(ax, funnel.people, "People by status", "people")
finish()

In [ ]:
# 4a Labels for everyone who reached a computer and mathematical job
reached = switches[switches.target_bucket.notna()]
show(f"Everyone who reached a computer and mathematical job ({len(reached):,} people): origin by retraining",
     pd.crosstab(reached.origin.fillna("none"), reached.retrained.map({True: "retrained", False: "no tech or math training"}),
                 margins=True, margins_name="total"))
show("Level of the job at the switch", reached.first_level.value_counts().rename_axis("first_level").to_frame("people"))
in_view = reached.status.str.startswith("candidate")
data_counts = pd.DataFrame({
    "switched into data and analytics": reached.target_bucket.eq("data and analytics"),  # within tech: the role moved into
    "reached data and analytics at any point": reached.reached_data,
})
within_tech = reached.status.eq("candidate: changed roles within tech")
show("Data and analytics: everyone vs. the career-changer view", pd.DataFrame({
    "everyone who reached tech": data_counts.sum(), "career changers (view)": data_counts[in_view].sum(),
    "of which: changed roles within tech": data_counts[within_tech].sum(),
    "of which: retrained": data_counts[in_view & ~within_tech].sum()}))

**What the data shows**
- **Almost everyone has a readable job date** (17,950 of 17,952), up from 63% before the date fix.
- **238 career changers moved into data and analytics: 216 by changing roles within tech, only 22 by retraining.** That's why the retraining compass (4a.2b) shows almost no data and analytics.
- **15,145 people (84%) held a computer and mathematical job;** 2,554 of them are career changers (4a.1).

### 4a.1 Viable candidates

How many people the rules leave, and who they are in numbers. **Viable** means they passed the rules; the rules were spot-checked on random samples (4b/4c), not every person. Counts are of unique people (after 2b). Aggregates only.

In [ ]:
# 4a.1a How many: from every resume to viable candidates
POOLS = ["candidate: retrained, new grad from another field", "candidate: retrained, after working in another field",
         "candidate: changed roles within tech"]
viable = switches[switches.status.isin(POOLS)].copy()
viable["path"] = viable.status.str.removeprefix("candidate: ").str.removeprefix("retrained, ")
viable["background_area"] = np.where(viable.origin_type.eq("job"), viable.source_title, viable.source_study)
viable["background_area"] = viable.background_area.map(background_area)
viable.loc[viable.from_bucket.notna(), "background_area"] = "tech: " + viable.from_bucket.dropna()
viable["switch_year"] = viable.switch_month.map(lambda m: m.year if pd.notna(m) else np.nan)
viable["years_worked_before"] = viable.background_months / 12
viable["programs"] = education.groupby("person_id").size().reindex(viable.index, fill_value=0)

steps = pd.Series({
    "resumes in the dataset": N_BEFORE_DEDUPE,
    "unique people": len(switches),
    "with a readable job date": (switches.status != "no dated jobs").sum(),
    "ever held a computer and mathematical job": switches.target_bucket.notna().sum(),
    "viable career changers (total)": len(viable),
    **{f"  {p}": (viable.path == p).sum() for p in viable.path.value_counts().index},
})
show("From every resume to viable candidates",
     pd.DataFrame({"people": steps, "share of unique people": (steps / len(switches)).round(3)}))

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.barh(steps.index[::-1].str.strip(), steps.values[::-1], color=[ORANGE if s.startswith("  ") else BLUE for s in steps.index[::-1]], height=0.65)
for y, v in enumerate(steps.values[::-1]):
    ax.text(v, y, f" {v:,}", va="center", color=INK_2, fontsize=8.5)
ax.set(title="From every resume to viable candidates", xlabel="people (orange: the paths)")
ax.grid(axis="y", visible=False)
ax.margins(x=0.15)
finish()

**What it shows**
- **2,554 career changers (14% of unique people):** 1,848 changed roles within tech, 369 retrained after working in another field and 337 retrained as new grads from another field.
- **The biggest drop is from the 15,145 who reached tech:** most didn't change field or role bucket in a way the pools count.

In [ ]:
# 4a.1b Who they are, in numbers: summary statistics, distributions and correlations
METRICS = {
    "switch_year": "switch year",
    "years_worked_before": "years in previous field or role",
    "n_jobs_before": "jobs before the switch",
    "n_jobs": "jobs on the resume",
    "programs": "programs on the resume",
}
m = viable[list(METRICS)].rename(columns=METRICS).astype(float)
gap = viable.loc[viable.origin_type.eq("job"), "gap_months"].rename("gap before the switch, months (after-work path)")
both = pd.concat([m, gap], axis=1)
summary = both.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).T
summary["skew"] = both.skew()
show("Summary statistics, all viable candidates", summary.round(1))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.4), gridspec_kw={"width_ratios": [1, 1, 1.1]})
for ax, col, subset, note in [(axes[0], "switch year", m["switch year"], ""),
                              (axes[1], "years in previous field or role", m["years in previous field or role"][m["years in previous field or role"] > 0], "\n(only people with time before the switch)")]:
    x = subset.dropna()
    counts, edges, _ = ax.hist(x, bins="auto", color=BLUE, alpha=0.85)
    grid = np.linspace(edges[0], edges[-1], 200)
    ax.plot(grid, stats.norm.pdf(grid, x.mean(), x.std()) * len(x) * np.diff(edges).mean(), color=ORANGE,
            label="bell curve with the same mean and std")
    ax.set(title=f"{col.capitalize()} (n = {len(x):,}, skew {x.skew():.1f}){note}", xlabel=col, ylabel="people")
    ax.legend(fontsize=8)
heatmap(axes[2], m.loc[:, m.std() > 0].corr(method="spearman"), "Spearman correlation", fmt="{:.2f}", cmap="RdBu_r", vmin=-1, vmax=1)
finish()

**What it shows**
- **The typical switch is in 2014, after 2.4 years in the previous field or role** (medians; half switched between 2010 and 2016). Both are far from a bell curve (skew −1.5 and 3.7), so medians describe them better than means.
- **No strong correlations.** The strongest is −0.31: later switchers list fewer jobs, because their resumes have had less time after the switch. Everything else is under ±0.25, so the features don't repeat each other.
- **People who retrained after working took about 14 months (median) from their last other-field job to their first tech job;** a quarter started within a month, and about 1 in 10 started tech work before leaving the old job (negative gaps). A few long tails (up to 55 years in the previous field) stay in; medians aren't moved by them.

### 4a.2 Retrained into tech or math: where they came from, what they trained in, and where it led

One row per viable candidate, stating their **prior career**, the **field they trained in** (grouped by `TRAINING_FIELDS` in `src/career_switch.py`) and the program as written. The full table is saved to `data/profiles/` for browsing; everything printed here is aggregate except the preview in 4a.2d.

In [ ]:
# 4a.2a The retrained table, saved, with totals by path
RETRAINED_COLS = {
    "path": "path",
    "background_area": "original background (area)",
    "source_title": "main background job",
    "source_study": "field studied first",
    "retrain_field": "field trained into (grouped)",
    "retrain_study": "program trained in",
    "target_bucket": "first tech role (bucket)",
    "target_title": "first tech job title",
    "roles_reached": "tech roles reached",
    "switch_period": "when they switched",
    "background_months": "months worked in another field",
    "study_undated": "relied on an undated program",
}
FIELD, ROLE, AREA, WHEN = "field trained into (grouped)", "first tech role (bucket)", "original background (area)", "when they switched"
retrained = viable[viable.status.str.startswith("candidate: retrained")][list(RETRAINED_COLS)].rename(columns=RETRAINED_COLS).sort_values([FIELD, ROLE])
RETRAINED_PATH = PROFILES_DIR / "retrained_candidates.csv"
retrained.to_csv(RETRAINED_PATH)
print(f"saved {len(retrained):,} rows to {RETRAINED_PATH.relative_to(DATA_DIR.parent)}")

by_path = retrained.path.value_counts()
show("Retrained career changers by path", pd.concat([by_path, pd.Series({"total": by_path.sum()})])
     .rename_axis("path").to_frame("people").assign(share=lambda d: (d.people / by_path.sum()).round(3)))

# One bar for the total, split by path
fig, ax = plt.subplots(figsize=(9, 1.9))
left = 0
for (path, n), color in zip(by_path.items(), [BLUE, ORANGE]):
    ax.barh([0], [n], left=left, color=color, height=0.6)
    ax.text(left + n / 2, 0, f"{path}\n{n:,} ({n / by_path.sum():.0%})", ha="center", va="center",
            color="white", fontsize=9, fontweight="bold")
    left += n
ax.set(title=f"Retrained career changers: {by_path.sum():,} in total", xlim=(0, by_path.sum()), yticks=[], xlabel="people")
ax.grid(False)
for side in ("left", "top", "right"):
    ax.spines[side].set_visible(False)
finish()

In [ ]:
# 4a.2b Where they came from, what they trained in, and where it led
# Prior careers and first tech jobs are in the path tables below (their people column and "all retrained" row)
fig, ax = plt.subplots(figsize=(8, 4.2))
bar_h(ax, retrained[FIELD].value_counts(), "Field trained in", "people")
finish()

# Prior career → first tech job: where people from each career landed (the user's first step: they know their
# background). Rows: the area of their main job before the switch, or the field they studied first for new grads;
# largest first, "other" and "unknown" last. Same layout as the compass below and the within-tech table in 4a.3.
by_size = retrained[AREA].value_counts().index
area_order = [a for a in by_size if a not in ("other", "unknown")] + [a for a in ("other", "unknown") if a in by_size]
prior = path_table(retrained, AREA, ROLE, area_order, "prior career (main job before; field studied first for new grads)",
                   "first tech job after training", "all retrained")
show(f"Prior career → first tech job: % of all {len(retrained):,} retrained people, by the career they left and their "
     "first tech job", prior)

# Retraining compass, the past side (the second step: which training to take). Rows: the field people trained in, after a job or degree outside tech.
# Columns: their first tech job after training. Cells: % of all retrained people; both axes run from infrastructure to
# code to data (src/career_switch.py), so related fields and roles sit near each other.
compass = path_table(retrained, FIELD, ROLE, FIELD_ORDER, "field trained in (after a job or degree outside tech)",
                     "first tech job after training", "all retrained")
show(f"Retraining compass: % of all {len(retrained):,} retrained people, by the field they trained in and their first "
     "tech job (within-tech moves are in 4a.3)", compass)

programs = retrained["program trained in"].dropna().str.lower().str.strip().value_counts()
show("Most common programs trained in, as written (top 10, shared by at least 5 people): checks the grouping",
     programs[programs >= 5].head(10).rename_axis("program").to_frame("people"))

**What it shows**
- **Prior career shapes where people land.** Arts and media (52 people), humanities (46) and engineering (46) went mostly into software (about half or more of each); business (109) and administration and office (80) spread across networks, software and IT support; logistics and trades (35) went about 40% into networks, and military and public service (26) about 40% into IT support. Overall, first tech jobs were software (33%), networks and systems (25%), IT support (16%), databases (13%), security (9%), data and analytics (3%) and quality assurance (under 1%).
- **The biggest single paths (% of all 706 retrained people):** computer science → software (75 people, 10.6%), software and web development → software (72, 10.2%), networking → networks and systems (32, 4.5%). Information technology, the biggest field (235), spreads across every role.
- **Only 22 of the 706 retrained people (3%) started in data and analytics,** so its share is small in every training field. People reach data mostly by moving within tech (216, 4a.3), so the compass should point career changers to a first tech role on the way to data, not only to a data course.
- **115 people (16%) fall in "other":** their job titles matched no area, so they're still the biggest row, down from 157 (22%) after widening the area keywords (4a.2e lists the remaining common titles). They stay in: they're real career changers. The top programs as written are all computer science or information technology and are grouped correctly.

In [ ]:
# 4a.2c When: career-change dates by field, and whether the path changed over time
PERIODS = ["before 2000", "2000-2004", "2005-2009", "2010-2014", "2015-2019", "2020 or later", "not dated"]
when = pd.crosstab(retrained[FIELD], retrained[WHEN]).reindex(columns=PERIODS, fill_value=0)
when = when.loc[:, when.sum() > 0]  # drop empty periods
when = when.loc[when.sum(axis=1).sort_values(ascending=False).index]

fig, axes = plt.subplots(1, 2, figsize=(17, 0.42 * len(when) + 2.2), gridspec_kw={"width_ratios": [1.3, 1]})
heatmap(axes[0], when, "Field trained into × when they switched (people)")

retrained_pools = viable[viable.status.str.startswith("candidate: retrained")]  # the within-tech pool isn't about new grads
d = retrained_pools[["switch_year"]].assign(new_grad=retrained_pools.path.eq("new grad from another field").astype(int)).dropna()
yearly = d.groupby("switch_year").new_grad.agg(["mean", "size"])
axes[1].scatter(yearly.index, yearly["mean"], s=15 + 185 * yearly["size"] / yearly["size"].max(),
                color=BLUE, alpha=0.7, label="share each year (dot size = people)")
if d.new_grad.nunique() == 2:
    logit = LogisticRegression().fit(d[["switch_year"]] - 2010, d.new_grad)
    years = np.arange(d.switch_year.min(), d.switch_year.max() + 1)
    axes[1].plot(years, logit.predict_proba(pd.DataFrame({"switch_year": years - 2010}))[:, 1], color=ORANGE,
                 label=f"logistic fit: odds × {np.exp(5 * logit.coef_[0][0]):.2f} every 5 years")
axes[1].set(title="Share who retrained as new grads (vs. after working), by switch year",
            xlabel="year of the switch", ylabel="share new grads", ylim=(-0.02, 1.02))
axes[1].legend(fontsize=8)
finish()

**What it shows**
- **Most retrained switches are from 2010–2019 (524 of 706, 74%);** only one is dated 2020 or later. Switch dates are descriptive only and never filter anyone.
- **Some fields are recent:** 76% of software and web development switches and 61% of cybersecurity switches are from 2015–2019, while information technology and computer science go back further.
- **Retraining shifted toward people who'd already worked:** the odds of being a new grad fall by about 40% every 5 years (× 0.58). From about 2015, most people who retrained had worked in another field first.

In [ ]:
# 4a.2d Preview: the first rows of the retrained table (shows rows: clear before committing)
display(retrained.head(20))

**Widening the prior-career areas.** 4a.2e lists the most common job titles (or first fields of study, for new grads) among retrained people whose prior career is "other": no keyword in `BACKGROUND_AREAS` (`src/career_switch.py`) matched them. Tell Claude which career each common title belongs to, in your own words (for example "titles like X are healthcare"), or add keywords to `BACKGROUND_AREAS` yourself; then rerun from 4a. Changing areas never changes who is a career changer.

In [ ]:
# 4a.2e Unmatched prior careers: the most common titles in "other" (shows titles: clear before committing; never paste into AI tools)
other = retrained[retrained[AREA].eq("other")]
text = other["main background job"].where(other.path.eq("after working in another field"), other["field studied first"])
titles = text.dropna().str.lower().str.strip().value_counts()
shared = titles[titles >= 3]
print(f'{len(other):,} retrained people have the prior career "other"; {titles.size:,} distinct titles; '
      f"{shared.sum():,} people share one of the {shared.size:,} titles held by 3+ people")
display(shared.head(30).rename_axis("title or first field of study").to_frame("people"))

### 4a.3 Changed roles within tech: across and up

People already in tech who moved into a **different role bucket** after at least 12 months in their first one (for example IT support → data and analytics). For a user who already works in tech, these are the comparable paths: the compass for moving across or up. Each move is also labelled sideways or up, by comparing the level of the new job with the highest level held in the first role. Aggregates only.

In [ ]:
# 4a.3a From which tech role to which (% of everyone who moved; same layout as the compass in 4a.2b)
within = viable[viable.status == "candidate: changed roles within tech"]
if within.empty:
    print("No one changed roles within tech under the current rules.")
else:
    moves = path_table(within, "from_bucket", "target_bucket", ROLE_ORDER, "tech job before the move",
                       "tech job after the move", "all within tech")
    show(f"Changed roles within tech: % of all {len(within):,} people, by their tech job before and after the move "
         "(the diagonal is 0: a move means a different role)", moves)

    # Sideways or up: the level of the new job vs. the highest level held in the first role
    LEVELS = {"other": 0, "senior": 1, "manager or above": 2}
    step = np.sign(within.first_level.map(LEVELS) - within.from_level.map(LEVELS))
    direction = step.map({1: "up (to a higher level)", 0: "sideways (same level)", -1: "down (to a lower level)"})
    show("Moves within tech: sideways or up, by the role they moved into (% of each role's movers)",
         (100 * pd.crosstab(within.target_bucket, direction, normalize="index")).round(1)
         .reindex(ROLE_ORDER).dropna(how="all").assign(people=within.target_bucket.value_counts())
         .rename_axis(index="tech job after the move", columns="move"))
    show("Moves within tech: sideways or up, overall", direction.value_counts().rename_axis("move").to_frame("people")
         .assign(share=lambda d: (100 * d.people / d.people.sum()).round(1)))

**What it shows**
- **1,848 people changed roles within tech.** The largest moves (% of all 1,848): IT support → networks and systems (15.5%), networks and systems → IT support (11.1%; traffic runs both ways), IT support → security (8.9%), developers → databases (7.7%) and → data and analytics (7.5%).
- **216 moved into data and analytics,** mostly from software development (about 140) and databases (about 55). With the 22 who retrained into it, it's the second-smallest destination for career changers (238 of 2,554; quality assurance is smallest at 40), so the compass shows it with its real share and the model weights it in training.
- **Most moves are sideways, not up:** 79% stayed at the same level, 9% moved up to a senior or manager title, and 12% moved to a lower one. The share moving up is similar for every role (7–10%; quality assurance's 22% rests on 37 people). Levels come from title words (senior, lead, manager), so "down" often means a senior title followed by a plain one; read these as rough.

### 4b Review round

Set `POOL` (one of the three career-changer pools, or a rejection status from 4a to check whether a rule throws real switches away), then run 4c. Each run shows **5 random people** from that pool, new ones every time, with each person's **whole timeline** (titles and programs with dates; no employers or locations), every entry's label and the switch marked. The printed seed brings a sample back (`SEED = <number>`). Tell Claude which `person_id`s look right or wrong, and why, in your own words; don't paste the timelines. Aim for **about 20 people per pool** (4 runs).

**What "right" means:**

| Pool | Right if | Common mistakes to look for |
|:--|:--|:--|
| Retrained after working in another field | A real non-tech job (12+ months), then a tech or math program, then a tech job | The "other field" job was really technical; the program came after the tech job; a student job counted as a career |
| Retrained, new grad from another field | Studied a non-tech field first, then a tech or math program, then a tech job | The first degree was really technical; the order of the programs is wrong (undated programs) |
| Changed roles within tech | 12+ months in one tech role, then a genuinely different tech role | The same job retitled; the two roles are really the same bucket; the up/sideways label is wrong |
| A rejection status | The rule was right to reject them | A real career changer the rule threw away |

**Also check:** the prior career area (is "other" hiding a clear career?) and, for within-tech moves, whether "up", "sideways" or "down" matches the titles.

In [ ]:
# 4b Review settings
POOL = "candidate: retrained, new grad from another field"
# or "candidate: retrained, after working in another field", "candidate: changed roles within tech",
# or a rejection status from 4a (for example "changed jobs without tech or math training")
N = 5

In [ ]:
# 4c Five random people from POOL, new ones every run (shows rows: clear before committing)
SEED = int(np.random.default_rng().integers(1_000_000))  # replace with a printed number to see that sample again
pool = switches[switches.status == POOL]
sample = pool.sample(min(N, len(pool)), random_state=SEED)
print(f"{POOL}: {len(pool):,} people | seed {SEED}: rerun for new people, or set SEED = {SEED} to see these again")

edu_by = prepare_education(education)
for pid, s in sample.iterrows():
    tl = person_timeline(jobs[jobs.person_id == pid], edu_by.get(pid), s.switch_month)
    tl["switch"] = np.where(tl.target_job & tl.start.eq(s.switch_month), "<- switch", "")
    area = (f"tech: {s.from_bucket}" if pd.notna(s.from_bucket)
            else background_area(s.source_title if s.origin_type == "job" else s.source_study))
    move = (f"moved: {s.from_bucket} -> {s.target_bucket}, level {s.from_level} -> {s.first_level}" if pd.notna(s.from_bucket)
            else f"first tech role: {s.target_bucket} (level {s.first_level}) | trained into: {s.retrain_field}")
    print(f"\nperson {pid} | prior career area: {area} | {move} | all tech roles: {s.roles_reached}")
    display(tl[["kind", "text", "start", "end", "label", "dated", "switch"]])

**Spot-check results**

Jacob spot-checked random samples with the checklist in 4b (2026-10-09). The samples looked right, so the rules stand unchanged. Per-pool counts weren't recorded; a future check can add them here (counts only, never the timelines).

## 5. Freeze and save: the modeling table

One table for the ML notebook: **everyone who reached a computer and mathematical job** (labels, not rejections), with **pre-switch features only** and the **labels** the role model predicts. It holds more people than the 2,554 career changers on purpose: the others are extra training examples (for example, people who went straight from a business degree into data show which backgrounds lead there). Results are reported on career changers only (`career_changer`), and the ML notebook tests whether training on everyone beats training on career changers alone. Every column has a role:
- **feature:** known before the switch, so the model may use it
- **label:** what the model predicts, or which people results are reported on (`career_changer`)
- **outcome:** describes what happened at or after the switch; **never a feature**
- **descriptive:** for tables and charts only

Rerun this section after any change to the rules in `src/career_switch.py`.

In [ ]:
# 5a Build and save the modeling table, with its feature dictionary
COLUMNS = {  # column: (role, meaning)
    "origin": ("feature", "Original background: other-field job or study, tech or math, unclear, school only, short job, or none"),
    "from_level": ("feature", "Within-tech moves: highest level held in the first tech role, before the move (other, senior, manager or above); empty for everyone else"),
    "background_area": ("feature", "Broad area of the main background job or field studied; 'tech: <role>' for within-tech moves"),
    "years_before": ("feature", "Years in the previous field or role before the switch"),
    "n_jobs_before": ("feature", "Jobs before the first tech job"),
    "study_level_before": ("feature", "Highest program before the first tech job: 0 high school, 1 associate, 2 bachelor's, 3 master's or certificate, 4 doctorate"),
    "retrained": ("feature", "Trained into tech or math between the old field and the tech job"),
    "retrain_field": ("feature", "Field trained into (grouped)"),
    "from_bucket": ("feature", "Starting tech role, for moves within tech"),
    "study_undated": ("feature", "Relied on an undated program to order the timeline (a data-quality flag)"),
    "target_bucket": ("label", "Role bucket at the switch: what the role model predicts"),
    "pool": ("label", "Career-changer pool, or 'not a career changer'"),
    "career_changer": ("label", "In one of the three career-changer pools; results are reported on these people"),
    "roles_reached": ("outcome", "Every tech role held, including after the switch"),
    "reached_data": ("outcome", "Ever held a data and analytics role, including after the switch"),
    "first_level": ("outcome", "Level of the job at the switch: part of the outcome"),
    "switch_period": ("descriptive", "When the switch happened; a current user has no switch date"),
    "target_title": ("descriptive", "Job title at the switch"),
}
table = switches[switches.target_bucket.notna()].copy()
table["background_area"] = np.where(table.origin_type.eq("job"), table.source_title, table.source_study)
table["background_area"] = table.background_area.map(background_area)
table.loc[table.from_bucket.notna(), "background_area"] = "tech: " + table.from_bucket.dropna()
table["years_before"] = (table.background_months / 12).round(1)
table["career_changer"] = table.status.str.startswith("candidate")
table["pool"] = np.where(table.career_changer, table.status.str.removeprefix("candidate: "), "not a career changer")
modeling = table[list(COLUMNS)]
MODELING_PATH = PROFILES_DIR / "modeling_table.pkl"
modeling.to_pickle(MODELING_PATH)
n_cc = int(modeling.career_changer.sum())
print(f"saved {len(modeling):,} people to {MODELING_PATH.relative_to(DATA_DIR.parent)}: {n_cc:,} career changers "
      f"(results are reported on these) + {len(modeling) - n_cc:,} others who reached tech (extra training examples only)")
with pd.option_context("display.max_colwidth", None):  # show each meaning in full
    show("Modeling table: every column and its role", pd.DataFrame(
        [(col, role, meaning) for col, (role, meaning) in COLUMNS.items()], columns=["column", "role", "meaning"]).set_index("column"))

In [ ]:
# 5b Preview: the first rows of the modeling table (shows rows: clear before committing; never paste into AI tools)
display(modeling.head())

## 6. Conclusion

**Frozen on 2026-10-09.** The final tables are `data/profiles/modeling_table.pkl` and `data/profiles/retrained_candidates.csv`; rerun the whole notebook after any change to `src/career_switch.py`.

**What we did.** Read 54,933 public resume records as timelines (17,952 unique people after removing duplicates; almost all with readable job dates), labelled everyone who reached a computer and mathematical job (15,145), and found **2,554 career changers** in three pools: 1,848 changed roles within tech, 369 retrained after working in another field, and 337 retrained as new grads from another field.

**What we found.**
- **Where retrained people land depends on where they came from:** software (33% of first tech jobs) and networks and systems (25%) lead overall, but arts, humanities and engineering backgrounds went mostly into software, while administration, business and logistics spread across IT support and networks. Training follows the same split: computer science and software development led to software; networking and cybersecurity to infrastructure; information technology spread across every role.
- **Data and analytics is reached through tech, not straight from a course:** 238 career changers moved into it, 216 of them from another tech role (mostly developers and DBAs) and only 22 by retraining (3% of retrained people). The compass should show a first tech role as a step on the way to data.
- **Most moves within tech are sideways:** 79% kept the same level and 9% moved up to a senior or manager title, so "moving up" needs its own evidence before it's part of the pitch.
- **Switches take time and have shifted:** the typical switch came after 2.4 years in the previous field (median), people who retrained after working took about 14 months from their last old-field job to their first tech job, and retraining has shifted from new grads toward people who'd already worked.

**Handoff.** The modeling table (section 5) goes to the ML notebook: pre-switch features, the role at the switch as the label, and outcome columns marked. It holds all 15,145 people who reached tech: the 2,554 career changers, on whom results are reported, plus 12,591 others as extra training examples, with origin as a feature. The ML notebook keeps whichever training set predicts career changers' roles better. The prior career and training tables (4a.2b) and the within-tech table (4a.3) are the past side of the retraining compass; EDA_SDA_2 adds each role's 2026 openings, pay and closing speed.

**Limits.**
- The rules were spot-checked on random samples (4b/4c), not every person, and the resumes were converted by OCR and an LLM.
- 16% of retrained people have a prior career of "other" (titles that matched no area), and job levels are read from title words, so the level split is rough.
- The resumes show only people who made a switch, so the tables show where switchers went, never how likely a switch is to work.
- The switches span two decades, so the 2026 listings supply today's openings, pay and timing.